# Assignment 1: Constraint Mapping Across Market, Dayzer, and Panorama

This notebook builds a mapping table across three PJMISO constraint sources: **Market**, **Dayzer**, and **Panorama**.

The task is treated as a name-translation problem across three datasets that describe the same type of grid object with different conventions. In this notebook, one constraint record is interpreted as a pair:

$$
\text{constraint} = (\text{monitored facility},\ \text{contingency})
$$

The final output uses the Market list as the anchor table. For each Market row, the notebook finds the closest Dayzer and Panorama rows, then attaches confidence labels and review reasons so that uncertain matches are visible instead of hidden.

## Method Overview

The matching process has five stages.

1. **Parse each source into comparable fields.**  
   Market already has separate `CONSTRAINT` and `CONTINGENCY` fields. Dayzer mostly stores both parts in one `NAME` field, so the notebook splits it into a facility part and a contingency part. Panorama has separate `Monitored Facility` and `Contingency Name` fields and sometimes includes a useful short facility alias in parentheses.

2. **Normalize text before comparing names.**  
   The same object may be written with different case, punctuation, spacing, voltage formatting, abbreviations, and base-case labels. The notebook applies a common text normalizer $N(\cdot)$ so that superficial formatting differences have less effect.

3. **Build a weighted matching string.**  
   Because a constraint is defined by a facility and a contingency, the search string combines both. The contingency text is repeated to give it more weight:

$$
s_i = N(f_i)\ \oplus\ \underbrace{N(q_i)\ \oplus\ N(q_i)\ \oplus\ N(q_i)}_{\text{extra contingency weight}}
$$

where $f_i$ is the facility text, $q_i$ is the contingency text, and $\oplus$ means string concatenation.

4. **Find the nearest candidate by text similarity.**  
   Each search string is converted into a character n-gram TF-IDF (Term Frequency–Inverse Document Frequency) vector , and the closest Dayzer/Panorama row is selected by cosine similarity:

$$
\operatorname{sim}(x,y) = \frac{x^\top y}{\lVert x\rVert_2\lVert y\rVert_2}
$$

$$
j^* = \arg\max_j\operatorname{sim}(v_i, u_j)
$$

where $v_i$ is the Market vector and $u_j$ is a candidate vector from Dayzer or Panorama.

5. **Assign confidence and review fields.**  
   The notebook does not treat every nearest neighbor as a confirmed match. It adds exact/partial contingency checks, source-side status labels, voltage checks, duplicate-target flags, interface flags, and final review reasons.

## Rules Used to Interpret Constraint Names

The assignment says that constraints are defined by `(facility, contingency)` pairs, but the three sources do not use the same naming style. The following rules are used to make the matching closer to the grid meaning of the names rather than only their raw spelling.

- **Facility and contingency are handled separately when possible.**  
  A candidate with the same facility but a different contingency may still be a different constraint. For that reason, contingency agreement is checked explicitly after the text match.

- **Base-case labels are normalized.**  
  `ACTUAL`, `BASE`, `NONE`, and similar labels are treated as no-contingency or base-case states. This helps match Dayzer rows such as `...:ACTUAL` with Panorama rows that may use `BASE`.

- **Partial contingency matching is allowed but kept below exact matching.**  
  Some Dayzer names encode complex outages, for example a double-contingency string. To avoid marking all such rows as complete mismatches, the notebook compares significant contingency tokens. A simplified version of the overlap rule is:

$$
\operatorname{overlap}(a,b)=\frac{|T(a)\cap T(b)|}{\min(|T(a)|,|T(b)|)}
$$

where $T(a)$ and $T(b)$ are the sets of significant tokens after removing generic words and numeric-only tokens. A partial match requires meaningful shared tokens; it is not treated as equal to exact contingency agreement.

- **Voltage mismatch is a warning sign.**  
  Similar station names at different voltage levels can refer to different equipment. If both sides contain voltage values and they do not overlap, the match is downgraded.

- **Interface and transfer constraints are flagged.**  
  Names containing terms such as `INTERFACE`, `TRANSFER`, or `WHEEL` may describe aggregate transfer limits rather than a single line or transformer. These rows are still kept in the output, but they are flagged for careful review.

- **Duplicate targets are flagged rather than automatically rejected.**  
  If multiple Market rows map to the same Dayzer or Panorama row, it may mean that one source is more aggregated than another. It may also mean that the nearest-neighbor match is ambiguous. The notebook keeps these rows and records the issue in `review_reason`.

- **Panorama date windows are audit information.**  
  Panorama includes `Earliest` and `Latest` fields, while Market and Dayzer do not provide directly comparable date windows. Therefore, the date check is used as a warning flag, not as a hard filter.

## Design Choice: Use Market as the Anchor List

This notebook answers the following question:

> For each Market constraint, what are the closest Dayzer and Panorama expressions of the same constraint?

That choice gives one output row per Market row. It is suitable for the required output format because the required CSV fields are `market_constraint`, `dayzer_constraint`, and `pano_constraint`.

This choice also has a limitation: the notebook does **not** build a full union of all Dayzer-only and Panorama-only constraints. Extra rows that exist only in Dayzer or only in Panorama are outside this output table. They are mentioned in the limitations section of the report.

## Step 1: Imports

This cell imports the packages used in the notebook.

- `pandas` and `numpy` are used for tabular data processing.
- `re` is used for parsing and normalizing constraint names.
- `TfidfVectorizer` converts normalized strings into character n-gram vectors.
- `NearestNeighbors` finds the closest candidate in Dayzer or Panorama for each Market row.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors


## Step 2: Define File Paths

This cell locates the three input files and defines the output folder. The path finder supports several common layouts so the notebook can run either from this assignment folder or from a project root.

Required input files:

- `Market PJMISO constraint list.csv`
- `Dayzer PJMISO constraint list.csv`
- `Pano PJMISO constraint list.csv`

The notebook writes the final matched CSV and summary report into an `output/assignment1_constraint_mapping` folder.

In [ ]:
REQUIRED_INPUT_FILES = [
    "Market PJMISO constraint list.csv",
    "Dayzer PJMISO constraint list.csv",
    "Pano PJMISO constraint list.csv",
]


def find_submission_root() -> Path:
    """Find the submission root using only relative candidate paths."""
    candidates = [
        Path("."),
        Path(".."),
        Path("../.."),
        Path("ecesis-assignment-submission"),
        Path("../ecesis-assignment-submission"),
        Path("../../ecesis-assignment-submission"),
        Path("Summer2026-main/ecesis-assignment-submission"),
    ]
    for root in candidates:
        data_dir = root / "data" / "Assignment 1 - constraint_mapping"
        if all((data_dir / filename).exists() for filename in REQUIRED_INPUT_FILES):
            return root

    searched = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Could not find the submission root. Searched relative paths:\n{searched}")


PROJECT_ROOT = find_submission_root()
DATA_DIR = PROJECT_ROOT / "data" / "Assignment 1 - constraint_mapping"
ASSIGNMENT_DIR = PROJECT_ROOT / "assignment1_constraint_mapping"
OUTPUT_DIR = ASSIGNMENT_DIR / "outputs"
REPORT_DIR = ASSIGNMENT_DIR / "report"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

MARKET_PATH = DATA_DIR / "Market PJMISO constraint list.csv"
DAYZER_PATH = DATA_DIR / "Dayzer PJMISO constraint list.csv"
PANO_PATH = DATA_DIR / "Pano PJMISO constraint list.csv"

MATCHED_CSV = OUTPUT_DIR / "assignment1_matched_results.csv"
REPORT_PATH = REPORT_DIR / "assignment1_summary_report.md"

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")




## Step 3: Load Raw Data

This cell loads the three source files into dataframes. At this stage the tables are intentionally left unchanged so the raw schemas can be inspected before any parsing or cleaning.

The row counts are important because the three sources are not expected to have the same number of rows. Different vendors may include different historical windows, interface constraints, and source-specific records.

In [3]:
market = pd.read_csv(MARKET_PATH)
dayzer = pd.read_csv(DAYZER_PATH)
pano = pd.read_csv(PANO_PATH)

print(f"Market rows: {len(market):,}, columns: {len(market.columns):,}")
print(f"Dayzer rows: {len(dayzer):,}, columns: {len(dayzer.columns):,}")
print(f"Panorama rows: {len(pano):,}, columns: {len(pano.columns):,}")


Market rows: 5,230, columns: 6
Dayzer rows: 13,813, columns: 2
Panorama rows: 21,963, columns: 5


## Step 4: Inspect Raw Schemas

This cell prints column names and previews the first few rows of each source. The purpose is to confirm how each source stores facility and contingency information.

Expected structure:

| Source | Facility field | Contingency field | Notes |
|---|---|---|---|
| Market | `CONSTRAINT` | `CONTINGENCY` | Already close to a facility-contingency pair |
| Dayzer | inside `NAME` | inside `NAME` | Often separated by a colon, but not always perfectly structured |
| Panorama | `Monitored Facility` | `Contingency Name` | Also includes `Earliest` and `Latest` validity fields |

This step justifies why the notebook needs source-specific parsing instead of direct exact matching.

In [4]:
print("Market columns:", market.columns.tolist())
print("Dayzer columns:", dayzer.columns.tolist())
print("Panorama columns:", pano.columns.tolist())

display(market.head(3))
display(dayzer.head(3))
display(pano.head(3))


Market columns: ['CONSTRAINT', 'CONTINGENCY', 'TOZONE', 'REPORTEDNAME', 'CONSTRAINTID', 'CONTINGENCYID']
Dayzer columns: ['CID', 'NAME']
Panorama columns: ['PID', 'Monitored Facility', 'Contingency Name', 'Earliest', 'Latest']


,CONSTRAINT,CONTINGENCY,TOZONE,REPORTEDNAME,CONSTRAINTID,CONTINGENCYID
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV,L500.CONASTONE-PEACHBOTTOM.5012,NaN,NOTTINGHM 2-3 SER DEV A 230 KV,10002384305,10002493264
1,LENOX 115 KV LENOX-NMESHOPP NML 1090,L230.ETOWANDA-HILLSIDE.2002 [NYISO],PENELEC,LENOX-NMESHOPP NML 1090 B 115 KV,10000566138,10001865201
2,EASTON 69 KV EAS-EMU,ACTUAL,DPL,EASTON 69 KV EAS-EMU,10004072634,10000680485


,CID,NAME
0,1,Eastern Interface
1,2,Central Interface
2,3,Western Interface


,PID,Monitored Facility,Contingency Name,Earliest,Latest
0,1,CHIC_AVE 138KV - PRAXAIR3 138KV (CHIC_AVE 138 ...,L765.Dumont-WiltonCenter.11215,2017-06-29 00:00:00-04:00,2026-04-28 23:00:00-04:00
1,2,177 BURN 345KV - MUNSTER2 345KV (177 BURN 345 ...,L765.Dumont-WiltonCenter.11215,2016-11-23 03:00:00-05:00,2026-04-28 23:00:00-04:00
2,3,CONASTON 500KV - CONASTON 230KV (CONASTON 500 ...,L500.Brighton-Conastone.5011,2021-02-17 06:00:00-05:00,2026-04-28 23:00:00-04:00


## Step 5: Validate Required Columns

This cell checks that each input file contains the columns used later in the notebook. The check is included to fail early with a clear error message if the files are renamed or if a different version of the dataset is used.

The matching logic depends on these fields:

$$
\text{Market: }(\texttt{CONSTRAINT},\texttt{CONTINGENCY},\texttt{CONSTRAINTID},\texttt{CONTINGENCYID})
$$

$$
\text{Dayzer: }(\texttt{CID},\texttt{NAME})
$$

$$
\text{Panorama: }(\texttt{PID},\texttt{Monitored Facility},\texttt{Contingency Name},\texttt{Earliest},\texttt{Latest})
$$

In [5]:
REQUIRED_COLUMNS = {
    "market": ["CONSTRAINTID", "CONTINGENCYID", "CONSTRAINT", "CONTINGENCY", "REPORTEDNAME"],
    "dayzer": ["CID", "NAME"],
    "pano": ["PID", "Monitored Facility", "Contingency Name"],
}


def require_columns(df: pd.DataFrame, required: list[str], source_name: str) -> None:
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise KeyError(f"{source_name} is missing required columns: {missing}")


require_columns(market, REQUIRED_COLUMNS["market"], "Market")
require_columns(dayzer, REQUIRED_COLUMNS["dayzer"], "Dayzer")
require_columns(pano, REQUIRED_COLUMNS["pano"], "Panorama")


## Step 6: Source-Specific Interpretation

Before matching, each source is interpreted as a `(facility, contingency)` pair.

- In Market, `CONSTRAINT` is treated as the monitored facility and `CONTINGENCY` as the contingency.
- In Dayzer, `NAME` is split into a facility part and a contingency part when a colon is present. Some rows are interface-style names and may not have a clean contingency.
- In Panorama, `Monitored Facility` and `Contingency Name` are already separated. The text inside parentheses is also extracted because it often contains a shorter alias closer to Market or Dayzer naming.

This source-specific interpretation is the first step of building a translation table between the three naming systems.

## Step 7: Text Normalization Functions

Raw names cannot be compared directly because the same object can be written with different punctuation, case, abbreviations, and spacing. This cell defines the function $N(t)$ that maps raw text into a normalized string.

Examples of normalization:

- `345 KV`, `345KV`, and `345 kV` are made comparable.
- separators such as `_`, `-`, `/`, `.`, and extra spaces are reduced to consistent spacing.
- common base-case labels such as `ACTUAL` and `BASE` are normalized together.
- common spelling variants are replaced where useful.

The goal is not to erase all information. The goal is to remove formatting noise while keeping meaningful tokens such as station names, voltage levels, circuit names, and contingency names.

In [6]:
COMMON_NAME_REPLACEMENTS = {
    "WILTON CENTER": "WILTONCENTER",
    "BLACK OAK": "BLACKOAK",
    "MT STORM": "MTSTORM",
    "ELECTRIC JUNCTION": "ELECTRICJUNCTION",
    "TANNERS CREEK": "TANNERSCREEK",
    "SOUTH CANTON": "SOUTHCANTON",
    "ERIE WEST": "ERIEWEST",
    "ERIE SOUTH": "ERIESOUTH",
    "NEW CARLISLE": "NEWCARLISLE",
    "PEACH BOTTOM": "PEACHBOTTOM",
    "ROCK SPRINGS": "ROCKSPRINGS",
    "HOPE CREEK": "HOPECREEK",
    "NORTH ANNA": "NORTHANNA",
    "POSSUM POINT": "POSSUMPOINT",
    "MIDDLE TAP": "MIDDLETAP",
    "FOUR RIVERS": "FOURRIVERS",
    "LAKE NELSON": "LAKENELSON",
}


def normalize_text(value: object) -> str:
    """Normalize grid names while preserving useful electrical identifiers."""
    if pd.isna(value):
        return ""

    text = str(value).upper().replace("_", " ")
    text = text.replace("&", " AND ")

    for old_text, new_text in COMMON_NAME_REPLACEMENTS.items():
        text = text.replace(old_text, new_text)

    text = re.sub(r"(\d+)\s*K\s*V", r"\1 KV ", text)
    text = re.sub(r"(?<=\d)(?=[A-Z])", " ", text)
    text = re.sub(r"(?<=[A-Z])(?=\d)", " ", text)
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    text = re.sub(r"\b[0-9]+ (LN|XF)\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_contingency(value: object) -> str:
    """Normalize contingency names and align common no-contingency labels."""
    text = normalize_text(value)
    if text in {"BASE", "ACTUAL", "NO CONTINGENCY", "NONE"}:
        return "ACTUAL"
    if text == "UNKNOWN":
        return ""
    return text


## Step 8: Dayzer Parsing Function

Dayzer stores many constraints in one `NAME` field. A common pattern is:

```text
facility : contingency
```

For example:

```text
FALCONER_115 KV_FAL-WAR:ACTUAL
```

is parsed as:

```text
facility    = FALCONER_115 KV_FAL-WAR
contingency = ACTUAL
```

The parser uses the **first** colon as the separator. This is a practical choice because some Dayzer contingency strings contain additional colons, such as double-contingency descriptions. Those more complex strings are handled later by the partial contingency matching step.

In [7]:
def split_dayzer_name(value: object) -> tuple[str, str]:
    """Split a Dayzer NAME into facility and contingency components."""
    if pd.isna(value):
        return "", ""

    text = str(value)
    if ":" in text:
        facility, contingency = text.split(":", 1)
        return facility.strip(), contingency.strip()
    return text.strip(), ""


## Step 9: Panorama Alias Extraction Function

Panorama facility names often contain a detailed name followed by a shorter alias in parentheses. For example:

```text
NMESHOPP 115KV - LENOX 115KV (LENOX 115 KV LEN-NME)
```

The text inside parentheses can be closer to the Market or Dayzer naming convention than the full Panorama name. This cell extracts that alias and later includes it in the Panorama search text.

In [8]:
def extract_pano_short_name(value: object) -> str:
    """Extract the last parenthetical alias from a Panorama monitored facility name."""
    if pd.isna(value):
        return ""

    matches = re.findall(r"\(([^()]*)\)", str(value))
    if matches:
        return matches[-1].strip()
    return str(value).strip()


## Step 10: Weighted Search Text Function

This cell defines the string used for TF-IDF matching. The search text combines normalized facility and normalized contingency fields.

Because the contingency is a key part of the constraint identity, it is repeated in the search text:

$$
s_i = N(f_i) + 3N(q_i)
$$

This does not change the raw data. It only changes the text representation used by the vectorizer, making contingency differences more important in the similarity score.

In [9]:
def build_search_text(facility_norm: str, contingency_norm: str) -> str:
    """Build a weighted text representation for matching."""
    parts = [
        facility_norm,
        facility_norm,
        contingency_norm,
        contingency_norm,
        contingency_norm,
    ]
    return " ".join(part for part in parts if part).strip()


## Step 11: Prepare Market Working Table

This cell creates the Market working table with standardized columns.

Market is used as the anchor list. Each Market row becomes one output row, and the notebook searches for the closest Dayzer and Panorama expressions of that Market constraint.

The canonical ID is not created yet. At this point, the cell only prepares the cleaned facility text, cleaned contingency text, and weighted search text used for matching.

In [10]:
market_clean = market.copy()
market_clean["market_constraint"] = (
    market_clean["CONSTRAINT"].fillna("").astype(str)
    + " : "
    + market_clean["CONTINGENCY"].fillna("").astype(str)
)
market_clean["facility_raw"] = (
    market_clean["REPORTEDNAME"].fillna("").astype(str)
    + " | "
    + market_clean["CONSTRAINT"].fillna("").astype(str)
)
market_clean["facility_norm"] = market_clean["facility_raw"].map(normalize_text)
market_clean["cont_norm"] = market_clean["CONTINGENCY"].map(normalize_contingency)
market_clean["search_text"] = market_clean.apply(
    lambda row: build_search_text(row["facility_norm"], row["cont_norm"]), axis=1
)

market_clean[["market_constraint", "facility_norm", "cont_norm", "search_text"]].head(3)


,market_constraint,facility_norm,cont_norm,search_text
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV : L500.C...,NOTTINGHM 2 3 SER DEV A 230 KV NOTTINGH 230 KV...,L 500 CONASTONE PEACHBOTTOM 5012,NOTTINGHM 2 3 SER DEV A 230 KV NOTTINGH 230 KV...
1,LENOX 115 KV LENOX-NMESHOPP NML 1090 : L230.ET...,LENOX NMESHOPP NML 1090 B 115 KV LENOX 115 KV ...,L 230 ETOWANDA HILLSIDE 2002 NYISO,LENOX NMESHOPP NML 1090 B 115 KV LENOX 115 KV ...
2,EASTON 69 KV EAS-EMU : ACTUAL,EASTON 69 KV EAS EMU EASTON 69 KV EAS EMU,ACTUAL,EASTON 69 KV EAS EMU EASTON 69 KV EAS EMU EAST...


## Step 12: Prepare Dayzer Working Table

This cell parses and normalizes Dayzer records.

The main challenge is that Dayzer uses a compact `NAME` field. Some rows are standard facility-contingency records, while others are interface or transfer names. The notebook keeps all rows because the assignment asks for cross-source matching, but later cells assign review flags when the Dayzer candidate is uncertain.

In [11]:
dayzer_clean = dayzer.copy()
dayzer_parts = dayzer_clean["NAME"].apply(lambda value: pd.Series(split_dayzer_name(value)))
dayzer_clean["facility_raw"] = dayzer_parts[0]
dayzer_clean["cont_raw"] = dayzer_parts[1]
dayzer_clean["dayzer_constraint"] = dayzer_clean["NAME"].astype(str)
dayzer_clean["facility_norm"] = dayzer_clean["facility_raw"].map(normalize_text)
dayzer_clean["cont_norm"] = dayzer_clean["cont_raw"].map(normalize_contingency)
dayzer_clean["search_text"] = dayzer_clean.apply(
    lambda row: build_search_text(row["facility_norm"], row["cont_norm"]), axis=1
)

dayzer_clean[["dayzer_constraint", "facility_raw", "cont_raw", "facility_norm", "cont_norm"]].head(3)


,dayzer_constraint,facility_raw,cont_raw,facility_norm,cont_norm
0,Eastern Interface,Eastern Interface,,EASTERN INTERFACE,
1,Central Interface,Central Interface,,CENTRAL INTERFACE,
2,Western Interface,Western Interface,,WESTERN INTERFACE,


## Step 13: Prepare Panorama Working Table

This cell prepares Panorama records by combining three pieces of information:

1. the full `Monitored Facility`,
2. the short alias extracted from parentheses,
3. the `Contingency Name`.

The Panorama search text includes both the full facility name and the alias because either one may be closer to Market or Dayzer wording. The date fields are kept for a later audit flag.

In [12]:
pano_clean = pano.copy()
pano_clean["facility_short"] = pano_clean["Monitored Facility"].map(extract_pano_short_name)
pano_clean["facility_raw"] = (
    pano_clean["facility_short"].fillna("").astype(str)
    + " | "
    + pano_clean["Monitored Facility"].fillna("").astype(str)
)
pano_clean["pano_constraint"] = (
    pano_clean["Monitored Facility"].fillna("").astype(str)
    + " : "
    + pano_clean["Contingency Name"].fillna("").astype(str)
)
pano_clean["facility_norm"] = pano_clean["facility_raw"].map(normalize_text)
pano_clean["cont_norm"] = pano_clean["Contingency Name"].map(normalize_contingency)
pano_clean["search_text"] = pano_clean.apply(
    lambda row: build_search_text(row["facility_norm"], row["cont_norm"]), axis=1
)

pano_clean[["pano_constraint", "facility_short", "facility_norm", "cont_norm"]].head(3)


,pano_constraint,facility_short,facility_norm,cont_norm
0,CHIC_AVE 138KV - PRAXAIR3 138KV (CHIC_AVE 138 ...,CHIC_AVE 138 KV CHI-PRA2,CHIC AVE 138 KV CHI PRA 2 CHIC AVE 138 KV PRAX...,L 765 DUMONT WILTONCENTER 11215
1,177 BURN 345KV - MUNSTER2 345KV (177 BURN 345 ...,177 BURN 345 KV BUR-MUN1,177 BURN 345 KV BUR MUN 1 177 BURN 345 KV MUNS...,L 765 DUMONT WILTONCENTER 11215
2,CONASTON 500KV - CONASTON 230KV (CONASTON 500 ...,CONASTON 500 KV 500-4,CONASTON 500 KV 500 4 CONASTON 500 KV CONASTON...,L 500 BRIGHTON CONASTONE 5011


## Step 14: Preview Normalized Search Fields

This cell displays a few examples after parsing and normalization. The purpose is to check whether the cleaned fields still preserve the important information:

- facility or interface name,
- voltage level if available,
- contingency name or base-case label,
- source-specific identifiers.

This is a useful sanity check before running the nearest-neighbor match.

In [13]:
print("Market examples")
display(market_clean[["market_constraint", "facility_norm", "cont_norm"]].head(5))

print("Dayzer examples")
display(dayzer_clean[["dayzer_constraint", "facility_norm", "cont_norm"]].head(5))

print("Panorama examples")
display(pano_clean[["pano_constraint", "facility_norm", "cont_norm"]].head(5))


Market examples


,market_constraint,facility_norm,cont_norm
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV : L500.C...,NOTTINGHM 2 3 SER DEV A 230 KV NOTTINGH 230 KV...,L 500 CONASTONE PEACHBOTTOM 5012
1,LENOX 115 KV LENOX-NMESHOPP NML 1090 : L230.ET...,LENOX NMESHOPP NML 1090 B 115 KV LENOX 115 KV ...,L 230 ETOWANDA HILLSIDE 2002 NYISO
2,EASTON 69 KV EAS-EMU : ACTUAL,EASTON 69 KV EAS EMU EASTON 69 KV EAS EMU,ACTUAL
3,SAYRECON230 KV SAY-SAY : ACTUAL,SAYRECON 230 KV SAY SAY SAYRECON 230 KV SAY SAY,ACTUAL
4,MOUN UGI 230 KV MOUN UGI 2 XFORMER : 230/66.MO...,MOUN UGI 230 KV 2 MOUN UGI 230 KV MOUN UGI 2 X...,230 66 MOUNTAIN T 1 SCTNLZ


Dayzer examples


,dayzer_constraint,facility_norm,cont_norm
0,Eastern Interface,EASTERN INTERFACE,
1,Central Interface,CENTRAL INTERFACE,
2,Western Interface,WESTERN INTERFACE,
3,BCPEP Interface,BCPEP INTERFACE,
4,Black Oak - Bedington Interface,BLACKOAK BEDINGTON INTERFACE,


Panorama examples


,pano_constraint,facility_norm,cont_norm
0,CHIC_AVE 138KV - PRAXAIR3 138KV (CHIC_AVE 138 ...,CHIC AVE 138 KV CHI PRA 2 CHIC AVE 138 KV PRAX...,L 765 DUMONT WILTONCENTER 11215
1,177 BURN 345KV - MUNSTER2 345KV (177 BURN 345 ...,177 BURN 345 KV BUR MUN 1 177 BURN 345 KV MUNS...,L 765 DUMONT WILTONCENTER 11215
2,CONASTON 500KV - CONASTON 230KV (CONASTON 500 ...,CONASTON 500 KV 500 4 CONASTON 500 KV CONASTON...,L 500 BRIGHTON CONASTONE 5011
3,SALTSPRG 138KV - MASURY 138KV (SALTSPRG 138 KV...,SALTSPRG 138 KV SAL MAS 1 SALTSPRG 138 KV MASU...,L 345 NILES SHENANGO
4,BURMA 115KV - PINEY 115KV (BURMA 115 KV BUR-PI...,BURMA 115 KV BUR PIN BURMA 115 KV PINEY 115 KV...,L 230 GLADE WARREN 2088


## Step 15: Define Nearest-Neighbor Text Matching

This cell defines the main matching function.

The text model uses character n-grams instead of word tokens. This is useful because constraint names often contain abbreviations, bus names, circuit IDs, voltage labels, and punctuation differences. Character n-grams allow strings such as `PEACHBOTTOM`, `PEACH-BOTTOM`, and `PeachBottom` to share many features after normalization.

For each Market row $i$, the notebook computes:

$$
\text{best Dayzer}(i)=\arg\max_j \operatorname{sim}(v_i,d_j)
$$

$$
\text{best Panorama}(i)=\arg\max_k \operatorname{sim}(v_i,p_k)
$$

The function also records whether the normalized contingency is an exact match or a partial match.

In [14]:
GENERIC_CONTINGENCY_TOKENS = {
    "ACTUAL", "BASE", "CONTINGENCY", "OUTAGE", "NONE", "NO", "UNKNOWN",
    "DBL", "DOUBLE", "SGL", "SINGLE", "TRIP", "LOSS", "NYISO", "MISO", "PJM", "KV",
}


def significant_contingency_tokens(norm_text: object) -> set[str]:
    """Extract meaningful outage/facility tokens from a normalized contingency string.

    Numeric IDs and generic words are removed so that a Dayzer double-contingency string such as
    `DBL ETOWANDA HILLSIDE LENOX NMESH` can partially match a Market contingency such as
    `L 230 ETOWANDA HILLSIDE 2002 NYISO` through the shared `ETOWANDA HILLSIDE` tokens.
    """
    if pd.isna(norm_text):
        return set()

    tokens = str(norm_text).upper().split()
    cleaned = set()
    for token in tokens:
        if token in GENERIC_CONTINGENCY_TOKENS:
            continue
        if token.isdigit():
            continue
        if len(token) < 3:
            continue
        # Drop leading circuit/voltage-style tokens such as L, 230, 500 after normalization.
        if re.fullmatch(r"[A-Z]?\d+", token):
            continue
        cleaned.add(token)
    return cleaned


def contingency_partial_match(source_cont_norm: object, target_cont_norm: object) -> bool:
    """Return True for meaningful partial contingency overlap.

    Exact equality is handled separately. Partial matching is intentionally conservative:
    it requires at least two shared significant tokens, or a long compacted contingency name
    contained inside the other name. This helps complex Dayzer names without making every
    loose token overlap a match.
    """
    left = "" if pd.isna(source_cont_norm) else str(source_cont_norm).upper().strip()
    right = "" if pd.isna(target_cont_norm) else str(target_cont_norm).upper().strip()

    if not left or not right:
        return False
    if left == right:
        return True

    left_tokens = significant_contingency_tokens(left)
    right_tokens = significant_contingency_tokens(right)
    if not left_tokens or not right_tokens:
        return False

    shared = left_tokens.intersection(right_tokens)
    smaller_size = min(len(left_tokens), len(right_tokens))
    if len(shared) >= 2 and len(shared) / smaller_size >= 0.50:
        return True

    left_compact = "".join(sorted(left_tokens))
    right_compact = "".join(sorted(right_tokens))
    shorter, longer = sorted([left_compact, right_compact], key=len)
    return len(shorter) >= 10 and shorter in longer


def nearest_match(source_df: pd.DataFrame, reference_df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """Match each source row to the nearest reference row using character n-gram TF-IDF."""
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)
    reference_matrix = vectorizer.fit_transform(reference_df["search_text"])
    source_matrix = vectorizer.transform(source_df["search_text"])

    neighbor_model = NearestNeighbors(n_neighbors=1, metric="cosine", algorithm="brute")
    neighbor_model.fit(reference_matrix)
    distances, indices = neighbor_model.kneighbors(source_matrix)

    best_indices = indices[:, 0]
    scores = 1 - distances[:, 0]
    matched_reference = reference_df.iloc[best_indices].reset_index(drop=True)

    id_column = "CID" if prefix == "dayzer" else "PID"
    constraint_column = "dayzer_constraint" if prefix == "dayzer" else "pano_constraint"

    source_cont = source_df["cont_norm"].reset_index(drop=True)
    target_cont = matched_reference["cont_norm"].reset_index(drop=True)
    exact_flags = source_cont.values == target_cont.values
    partial_flags = [
        (not bool(exact)) and contingency_partial_match(left, right)
        for exact, left, right in zip(exact_flags, source_cont, target_cont)
    ]

    matched = pd.DataFrame(index=source_df.index)
    matched[f"{prefix}_id"] = matched_reference[id_column].values
    matched[f"{prefix}_constraint"] = matched_reference[constraint_column].values
    matched[f"{prefix}_score"] = scores
    matched[f"{prefix}_contingency_exact"] = exact_flags
    matched[f"{prefix}_contingency_partial"] = partial_flags
    matched[f"{prefix}_facility_norm"] = matched_reference["facility_norm"].values
    matched[f"{prefix}_contingency_norm"] = matched_reference["cont_norm"].values
    return matched


## Step 16: Match Market to Dayzer

This cell applies the nearest-neighbor function to find one Dayzer candidate for each Market row.

The output of this step is not treated as automatically correct. It is only the best candidate under the text similarity model. Later cells decide whether this candidate is accepted, partial, reviewable, or low-confidence.

In [15]:
dayzer_matches = nearest_match(market_clean, dayzer_clean, "dayzer")
dayzer_matches.head(5)


,dayzer_id,dayzer_constraint,dayzer_score,dayzer_contingency_exact,dayzer_contingency_partial,dayzer_facility_norm,dayzer_contingency_norm
0,532145,NOTTINGH_230 KV_2-3:L500.Conastone-PeachBottom...,0.723973,True,False,NOTTINGH 230 KV 2 3,L 500 CONASTONE PEACHBOTTOM 5012
1,552352,LENOX_115 KV_LEN-NME:DBL:ETowanda-Hillside+Len...,0.713047,False,True,LENOX 115 KV LEN NME,DBL ETOWANDA HILLSIDE LENOX NMESH
2,520100,EASTON_69 KV_EAS-EMU:ACTUAL,0.978077,True,False,EASTON 69 KV EAS EMU,ACTUAL
3,512652,SAYRECON_230 KV_SAY-SAY:ACTUAL,0.990167,True,False,SAYRECON 230 KV SAY SAY,ACTUAL
4,550634,MOUN UGI_230 KV_1:230/66.Mountain.T2 (Sctnlz),0.875760,False,True,MOUN UGI 230 KV 1,230 66 MOUNTAIN T 2 SCTNLZ


## Step 17: Match Market to Panorama

This cell applies the same matching function to Panorama.

Panorama usually has more structured fields than Dayzer, so its match quality is expected to be higher on average. The notebook still applies the same exact/partial contingency checks and later audit rules to keep the treatment consistent.

In [16]:
pano_matches = nearest_match(market_clean, pano_clean, "pano")
pano_matches.head(5)


,pano_id,pano_constraint,pano_score,pano_contingency_exact,pano_contingency_partial,pano_facility_norm,pano_contingency_norm
0,15915,NOTTINGHM 2-3 SER DEV A 230 KV : L500.C...,0.975537,True,False,NOTTINGHM 2 3 SER DEV A 230 KV NOTTINGHM 2 3 S...,L 500 CONASTONE PEACHBOTTOM 5012
1,783,NMESHOPP 115KV - LENOX 115KV (LENOX 115 KV LEN...,0.804073,True,False,LENOX 115 KV LEN NME NMESHOPP 115 KV LENOX 115...,L 230 ETOWANDA HILLSIDE 2002 NYISO
2,1920,EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU...,0.946864,True,False,EASTON 69 KV EAS EMU EASTON 69 KV EMUNI 69 KV ...,ACTUAL
3,2035,SAYREVIL 230KV - SAYRECON 230KV (SAYRECON 230 ...,0.967807,True,False,SAYRECON 230 KV SAY SAY SAYREVIL 230 KV SAYREC...,ACTUAL
4,1060,MOUN UGI 69KV - MOUN UGI 230KV (MOUN UGI 230 K...,0.971633,True,False,MOUN UGI 230 KV 2 MOUN UGI 69 KV MOUN UGI 230 ...,230 66 MOUNTAIN T 1 SCTNLZ


## Step 18: Define Initial Quality Rules

This cell converts each source-side score into an initial label.

The initial label combines two ideas:

1. the cosine similarity score, and
2. whether the contingency is exact or partial.

Exact contingency agreement can support a high label even when facility text is abbreviated. Partial contingency agreement can support a medium label, but it is kept below exact agreement because it may indicate a related but not identical outage.

The combined `match_quality` is the stricter result of the Dayzer side and the Panorama side. If either side is weak, the overall row should not be called clean high confidence.

In [17]:
def classify_side(score: float, contingency_exact: bool, contingency_partial: bool = False) -> str:
    """Classify one source-side match before audit checks."""
    if contingency_exact and score >= 0.70:
        return "high"
    if contingency_exact and score >= 0.55:
        return "medium"

    # Partial contingency agreement is useful for complex Dayzer names such as
    # DBL:ETowanda-Hillside+Lenox-NMesh, but it should remain below exact agreement.
    if contingency_partial and score >= 0.60:
        return "medium"

    if score >= 0.88:
        return "high"
    if score >= 0.72:
        return "medium"
    return "review"


def combine_quality(dayzer_quality: str, pano_quality: str) -> str:
    """Combine Dayzer and Panorama side-level confidence into one initial label."""
    if "review" in {dayzer_quality, pano_quality}:
        return "review"
    if "medium" in {dayzer_quality, pano_quality}:
        return "medium"
    return "high"


## Step 19: Build the Initial Matched Table

This cell combines the Market anchor row with its best Dayzer and Panorama candidates.

The resulting table contains:

- raw matched constraint names,
- source IDs,
- similarity scores,
- exact and partial contingency flags,
- normalized facility and contingency fields,
- the initial `match_quality`.

At this point the table is a first-pass match table. It still needs source-side status fields and audit checks.

In [18]:
result = pd.DataFrame(
    {
        "market_constraint": market_clean["market_constraint"],
        "market_constraint_id": market_clean["CONSTRAINTID"],
        "market_contingency_id": market_clean["CONTINGENCYID"],
        "market_facility_norm": market_clean["facility_norm"],
        "market_contingency_norm": market_clean["cont_norm"],
    }
)

result = pd.concat([result, dayzer_matches, pano_matches], axis=1)

result["dayzer_match_quality"] = result.apply(
    lambda row: classify_side(
        row["dayzer_score"],
        row["dayzer_contingency_exact"],
        row["dayzer_contingency_partial"],
    ),
    axis=1,
)
result["pano_match_quality"] = result.apply(
    lambda row: classify_side(
        row["pano_score"],
        row["pano_contingency_exact"],
        row["pano_contingency_partial"],
    ),
    axis=1,
)
result["match_quality"] = result.apply(
    lambda row: combine_quality(row["dayzer_match_quality"], row["pano_match_quality"]), axis=1
)

result["dayzer_score"] = result["dayzer_score"].round(4)
result["pano_score"] = result["pano_score"].round(4)

result[[
    "market_constraint",
    "dayzer_constraint",
    "pano_constraint",
    "match_quality",
    "dayzer_score",
    "pano_score",
    "dayzer_contingency_exact",
    "dayzer_contingency_partial",
]].head(5)


,market_constraint,dayzer_constraint,pano_constraint,match_quality,dayzer_score,pano_score,dayzer_contingency_exact,dayzer_contingency_partial
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV : L500.C...,NOTTINGH_230 KV_2-3:L500.Conastone-PeachBottom...,NOTTINGHM 2-3 SER DEV A 230 KV : L500.C...,high,0.7240,0.9755,True,False
1,LENOX 115 KV LENOX-NMESHOPP NML 1090 : L230.ET...,LENOX_115 KV_LEN-NME:DBL:ETowanda-Hillside+Len...,NMESHOPP 115KV - LENOX 115KV (LENOX 115 KV LEN...,medium,0.7130,0.8041,False,True
2,EASTON 69 KV EAS-EMU : ACTUAL,EASTON_69 KV_EAS-EMU:ACTUAL,EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU...,high,0.9781,0.9469,True,False
3,SAYRECON230 KV SAY-SAY : ACTUAL,SAYRECON_230 KV_SAY-SAY:ACTUAL,SAYREVIL 230KV - SAYRECON 230KV (SAYRECON 230 ...,high,0.9902,0.9678,True,False
4,MOUN UGI 230 KV MOUN UGI 2 XFORMER : 230/66.MO...,MOUN UGI_230 KV_1:230/66.Mountain.T2 (Sctnlz),MOUN UGI 69KV - MOUN UGI 230KV (MOUN UGI 230 K...,medium,0.8758,0.9716,False,True


## Step 20: Add Source-Side Match Status

The nearest-neighbor model always returns one candidate, even when the candidate is weak. To avoid making weak candidates look confirmed, this cell adds separate status fields for Dayzer and Panorama.

Possible status values:

| Status | Meaning |
|---|---|
| `matched` | strong enough to treat as an accepted candidate |
| `partial_match` | meaningful partial contingency overlap, but not exact |
| `needs_review` | plausible candidate that should be checked manually |
| `unmatched_low_confidence` | nearest candidate is shown for transparency but should not be treated as a reliable match |

The companion fields `dayzer_match_note` and `pano_match_note` make low-confidence candidates explicit in the CSV.

In [19]:
def source_match_status(score: float, exact: bool, partial: bool, side_quality: str) -> str:
    """Assign source-side status so low-confidence candidates are not treated as accepted matches.

    The nearest-neighbor matcher always returns one candidate. A low status means
    the candidate remains visible for analyst review, but should not be treated
    as a confirmed source mapping.
    """
    if bool(exact) and score >= 0.70:
        return "matched"
    if bool(exact) and score >= 0.55:
        return "needs_review"
    if bool(partial) and score >= 0.60:
        return "partial_match"
    if score < 0.60:
        return "unmatched_low_confidence"
    if side_quality == "review":
        return "needs_review"
    return "needs_review"


def source_match_note(status: str, constraint: object) -> str:
    """Create a readable note showing whether the candidate is accepted or only shown for review."""
    candidate = "" if pd.isna(constraint) else str(constraint)
    if status == "unmatched_low_confidence":
        return f"UNMATCHED / low-confidence candidate: {candidate}"
    if status == "needs_review":
        return f"NEEDS REVIEW candidate: {candidate}"
    if status == "partial_match":
        return f"PARTIAL MATCH candidate: {candidate}"
    return candidate


result["dayzer_match_status"] = result.apply(
    lambda row: source_match_status(
        row["dayzer_score"],
        row["dayzer_contingency_exact"],
        row["dayzer_contingency_partial"],
        row["dayzer_match_quality"],
    ),
    axis=1,
)
result["pano_match_status"] = result.apply(
    lambda row: source_match_status(
        row["pano_score"],
        row["pano_contingency_exact"],
        row["pano_contingency_partial"],
        row["pano_match_quality"],
    ),
    axis=1,
)

result["dayzer_match_note"] = [
    source_match_note(status, constraint)
    for status, constraint in zip(result["dayzer_match_status"], result["dayzer_constraint"])
]
result["pano_match_note"] = [
    source_match_note(status, constraint)
    for status, constraint in zip(result["pano_match_status"], result["pano_constraint"])
]

result[[
    "market_constraint",
    "dayzer_constraint",
    "dayzer_match_status",
    "pano_constraint",
    "pano_match_status",
    "dayzer_score",
    "pano_score",
]].head(10)

,market_constraint,dayzer_constraint,dayzer_match_status,pano_constraint,pano_match_status,dayzer_score,pano_score
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV : L500.C...,NOTTINGH_230 KV_2-3:L500.Conastone-PeachBottom...,matched,NOTTINGHM 2-3 SER DEV A 230 KV : L500.C...,matched,0.7240,0.9755
1,LENOX 115 KV LENOX-NMESHOPP NML 1090 : L230.ET...,LENOX_115 KV_LEN-NME:DBL:ETowanda-Hillside+Len...,partial_match,NMESHOPP 115KV - LENOX 115KV (LENOX 115 KV LEN...,matched,0.7130,0.8041
2,EASTON 69 KV EAS-EMU : ACTUAL,EASTON_69 KV_EAS-EMU:ACTUAL,matched,EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU...,matched,0.9781,0.9469
3,SAYRECON230 KV SAY-SAY : ACTUAL,SAYRECON_230 KV_SAY-SAY:ACTUAL,matched,SAYREVIL 230KV - SAYRECON 230KV (SAYRECON 230 ...,matched,0.9902,0.9678
4,MOUN UGI 230 KV MOUN UGI 2 XFORMER : 230/66.MO...,MOUN UGI_230 KV_1:230/66.Mountain.T2 (Sctnlz),partial_match,MOUN UGI 69KV - MOUN UGI 230KV (MOUN UGI 230 K...,matched,0.8758,0.9716
5,GRACETON 230 KV GRACETON-SAFEHARB 2303 : L500....,GRACETON_230 KV_GRA-MANO:L500.Conastone-PeachB...,matched,GRACETON230 KV GRA-SAF : L500.Conastone-Peach...,matched,0.7812,0.9114
6,94 HAURD-11323 11323 B 138 KV : L345.NELSON-EL...,94 HAURD_138 KV_11323:ACTUAL,needs_review,94 HAURD 138KV - 11323 138KV (94 HAURD 138 KV ...,matched,0.7420,0.9560
7,BERGEN 230 KV BER-HUD : ACTUAL,BERGEN_230 KV_BER-HUD:ACTUAL,matched,BERGEN 230KV - HCS 230KV (BERGEN 230 KV BER-HU...,matched,0.9785,0.9424
8,LINE 69 KV MONR AE-VINELAND 0711-1 : L230.CUMB...,MONR AE_69 KV_MON-VINE:230/138.Cumberland.T2,partial_match,MONR AE 69KV - VINELAND 69KV (MONR AE 69 KV MO...,matched,0.7466,0.9400
9,GARDNERS 115 KV GARDNERS-TEXSEAST GAR-TEX : L1...,GARDNERS_115 KV_GAR-TEX:L115.MiddletownJct-Col...,matched,TEXSEAST 115KV - GARDNERS 115KV (GARDNERS 115 ...,matched,0.8665,0.9937


## Step 21: Add Canonical Constraint IDs

This cell creates a first-pass canonical identifier for each output row.

Because the Market table contains both `CONSTRAINTID` and `CONTINGENCYID`, the ID is built from both:

$$
\texttt{canonical\_constraint\_id}
= \texttt{PJMISO\_CANON\_}\{\texttt{CONSTRAINTID}\}\_\{\texttt{CONTINGENCYID}\}
$$

This is more appropriate than using `CONSTRAINTID` alone because the assignment defines a constraint as a facility-contingency pair. The same facility can appear under multiple contingencies, so the contingency ID must be part of the canonical identifier.

In [20]:
def build_canonical_id(row: pd.Series, position: int) -> str:
    """Create a stable internal ID for the monitored-facility plus contingency pair.

    Market CONSTRAINTID alone is not unique because the same monitored facility
    can appear under several contingencies.  Therefore the canonical first-pass
    ID combines Market CONSTRAINTID and CONTINGENCYID, which uniquely identify
    the facility-contingency pair in the Market anchor list.
    """
    raw_constraint_id = row.get("market_constraint_id", "")
    raw_contingency_id = row.get("market_contingency_id", "")

    constraint_id = re.sub(r"[^A-Za-z0-9]+", "", str(raw_constraint_id))
    contingency_id = re.sub(r"[^A-Za-z0-9]+", "", str(raw_contingency_id))

    if (
        constraint_id
        and contingency_id
        and constraint_id.upper() not in {"NAN", "NONE"}
        and contingency_id.upper() not in {"NAN", "NONE"}
    ):
        return f"PJMISO_CANON_{constraint_id}_{contingency_id}"

    return f"PJMISO_CANON_ROW_{position:05d}"


result.insert(
    0,
    "canonical_constraint_id",
    [build_canonical_id(row, position) for position, (_, row) in enumerate(result.iterrows(), start=1)],
)

print("Unique canonical IDs:", result["canonical_constraint_id"].nunique(), "of", len(result))
result[["canonical_constraint_id", "market_constraint_id", "market_contingency_id", "dayzer_id", "pano_id"]].head(5)

Unique canonical IDs: 5230 of 5230


,canonical_constraint_id,market_constraint_id,market_contingency_id,dayzer_id,pano_id
0,PJMISO_CANON_10002384305_10002493264,10002384305,10002493264,532145,15915
1,PJMISO_CANON_10000566138_10001865201,10000566138,10001865201,552352,783
2,PJMISO_CANON_10004072634_10000680485,10004072634,10000680485,520100,1920
3,PJMISO_CANON_10001822928_10000680485,10001822928,10000680485,512652,2035
4,PJMISO_CANON_10000465752_10016012236,10000465752,10016012236,550634,1060


## Step 22: Meaning of `match_quality`

`match_quality` is the **initial** confidence label based on text similarity and contingency agreement.

It answers this question:

> Based on normalized text matching alone, how strong does the Market-Dayzer-Panorama row look?

It does **not** yet account for all warning signs. For final interpretation, use `audited_match_quality`, which is created after the audit checks below.

## Step 23: Voltage Audit Functions

This cell extracts voltage levels such as `69`, `115`, `138`, `230`, `345`, and `500` kV from constraint names.

Voltage is not always available, so missing voltage is not treated as a mismatch. The logic is:

$$
\text{voltage status}=\begin{cases}
\text{unknown}, & V_1=\varnothing\text{ or }V_2=\varnothing\\
\text{match}, & V_1\cap V_2\ne\varnothing\\
\text{mismatch}, & V_1\cap V_2=\varnothing
\end{cases}
$$

where $V_1$ and $V_2$ are the extracted voltage sets from the two source names.

In [21]:
def extract_voltages(text: object) -> list[int]:
    """Extract common voltage levels from a constraint name."""
    if pd.isna(text):
        return []
    normalized = str(text).upper().replace("/", " ")
    matches = re.findall(r"\b(34|46|69|115|138|161|230|345|500|765)\s*K\s*V\b", normalized)
    return sorted({int(value) for value in matches})


def voltage_status(left: list[int], right: list[int]) -> str:
    """Return match, mismatch, or unknown based on voltage overlap."""
    if not left or not right:
        return "unknown"
    return "match" if set(left).intersection(right) else "mismatch"


def format_voltage_list(values: list[int]) -> str:
    """Format extracted voltage values for CSV output."""
    return "|".join(map(str, values))


## Step 24: Apply Voltage Audit

This cell applies the voltage extraction functions to Market, Dayzer, and Panorama names.

A voltage mismatch is a strong warning because similar station names at different voltage levels may refer to different physical equipment. An unknown voltage status is much weaker; it usually means the voltage was not written in one of the names.

In [22]:
market_voltage_lists = result["market_constraint"].map(extract_voltages)
dayzer_voltage_lists = result["dayzer_constraint"].map(extract_voltages)
pano_voltage_lists = result["pano_constraint"].map(extract_voltages)

result["dayzer_voltage_status"] = [
    voltage_status(left, right) for left, right in zip(market_voltage_lists, dayzer_voltage_lists)
]
result["pano_voltage_status"] = [
    voltage_status(left, right) for left, right in zip(market_voltage_lists, pano_voltage_lists)
]

result["market_voltage_kv"] = market_voltage_lists.map(format_voltage_list)
result["dayzer_voltage_kv"] = dayzer_voltage_lists.map(format_voltage_list)
result["pano_voltage_kv"] = pano_voltage_lists.map(format_voltage_list)

result[["market_constraint", "dayzer_voltage_status", "pano_voltage_status", "market_voltage_kv", "dayzer_voltage_kv", "pano_voltage_kv"]].head(5)


,market_constraint,dayzer_voltage_status,pano_voltage_status,market_voltage_kv,dayzer_voltage_kv,pano_voltage_kv
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV : L500.C...,unknown,match,230,,230
1,LENOX 115 KV LENOX-NMESHOPP NML 1090 : L230.ET...,unknown,match,115,,115
2,EASTON 69 KV EAS-EMU : ACTUAL,unknown,match,69,,69
3,SAYRECON230 KV SAY-SAY : ACTUAL,unknown,unknown,,,230
4,MOUN UGI 230 KV MOUN UGI 2 XFORMER : 230/66.MO...,unknown,match,230,,69|230


## Step 25: Duplicate Target Audit

This cell flags many-to-one matches.

A duplicate target occurs when multiple Market rows select the same Dayzer or Panorama row. This can happen for two different reasons:

1. the target source is more aggregated than Market, or
2. the text matcher is choosing the same generic-looking candidate for several different Market rows.

Because both explanations are possible, duplicate targets are kept in the output but recorded in `review_reason`.

In [23]:
result["dayzer_duplicate_match"] = result.duplicated("dayzer_id", keep=False)
result["pano_duplicate_match"] = result.duplicated("pano_id", keep=False)

print("Rows sharing a Dayzer target:", int(result["dayzer_duplicate_match"].sum()))
print("Rows sharing a Panorama target:", int(result["pano_duplicate_match"].sum()))


Rows sharing a Dayzer target: 1697
Rows sharing a Panorama target: 1060


## Step 26: Interface and Transfer Constraint Audit

This cell flags names that look like interface, transfer, wheel, or aggregate constraints.

These records can be valid constraints, but they are harder to match with the ordinary single-facility plus contingency assumption. For example, an `APSouth Interface` row may describe a transfer interface rather than one specific line. The notebook therefore keeps these rows but marks them for review.

In [24]:
def is_interface(text: object) -> bool:
    """Flag interface-style constraints that may not be ordinary facility-contingency pairs."""
    if pd.isna(text):
        return False

    upper = str(text).upper()
    interface_terms = [
        "INTERFACE",
        " TRANSFER ",
        " FLOWGATE ",
        "FLOWGATE",
        "PA-CENT",
        "WESTERN",
        "CENTRAL",
        "EASTERN",
    ]
    return any(term in f" {upper} " for term in interface_terms)


result["market_interface_flag"] = result["market_constraint"].map(is_interface)
result["dayzer_interface_flag"] = result["dayzer_constraint"].map(is_interface)
result["pano_interface_flag"] = result["pano_constraint"].map(is_interface)
result["interface_flag"] = result[["market_interface_flag", "dayzer_interface_flag", "pano_interface_flag"]].any(axis=1)

print("Interface-style rows:", int(result["interface_flag"].sum()))


Interface-style rows: 131


## Step 27: Panorama Validity Window Audit

Panorama includes `Earliest` and `Latest` fields. This cell attaches those dates to the matched output and creates a simple validity flag.

The date check is not used as a hard filter because Market and Dayzer do not provide directly comparable effective dates. Instead, it is kept as an audit field so the reviewer can see whether a Panorama match is active at the latest Panorama timestamp or appears to be historical.

In [25]:
pano_dates = pano_clean[["PID"]].copy().rename(columns={"PID": "pano_id"})

if {"Earliest", "Latest"}.issubset(pano_clean.columns):
    pano_dates["pano_earliest"] = pano_clean["Earliest"]
    pano_dates["pano_latest"] = pano_clean["Latest"]
else:
    pano_dates["pano_earliest"] = pd.NaT
    pano_dates["pano_latest"] = pd.NaT

result = result.drop(columns=[column for column in ["pano_earliest", "pano_latest"] if column in result.columns])
result = result.merge(pano_dates, on="pano_id", how="left")

earliest = pd.to_datetime(result["pano_earliest"], errors="coerce", utc=True)
latest = pd.to_datetime(result["pano_latest"], errors="coerce", utc=True)
pano_snapshot_time = latest.max() if latest.notna().any() else pd.NaT

if pd.isna(pano_snapshot_time):
    result["pano_validity_flag"] = "validity_window_unavailable"
else:
    result["pano_validity_flag"] = np.select(
        [
            earliest.isna() | latest.isna(),
            latest < pano_snapshot_time,
            earliest > pano_snapshot_time,
        ],
        [
            "missing_validity_window",
            "historical_window_before_latest_snapshot",
            "starts_after_latest_pano_snapshot",
        ],
        default="active_at_latest_pano_snapshot",
    )

result["pano_validity_flag"].value_counts(dropna=False)


pano_validity_flag
historical_window_before_latest_snapshot    5221
active_at_latest_pano_snapshot                 9
Name: count, dtype: int64

## Step 28: Build Human-Readable Review Reasons

This cell builds the `review_reason` field. The goal is to make every downgrade explainable.

The review reason can include score warnings, contingency mismatch, partial contingency, voltage mismatch, duplicate target, interface-style constraint, Panorama validity issue, or low-confidence source-side status.

This field is important because a medium or review label by itself is not very informative. `review_reason` explains what should be checked.

In [26]:
def join_reasons(reasons: list[str]) -> str:
    """Join unique reasons into a stable semicolon-separated string."""
    return ";".join(dict.fromkeys(reason for reason in reasons if reason))


def cross_source_confirmed_high(row: pd.Series) -> bool:
    """Return True when two exact contingencies and strong Panorama evidence support a high label."""
    return (
        bool(row["dayzer_contingency_exact"])
        and bool(row["pano_contingency_exact"])
        and row["dayzer_score"] >= 0.70
        and row["pano_score"] >= 0.95
        and row["dayzer_voltage_status"] != "mismatch"
        and row["pano_voltage_status"] != "mismatch"
        and not bool(row["interface_flag"])
    )


def add_score_reason(reasons: list[str], prefix: str, score: float, partial: bool, cross_confirmed: bool) -> None:
    """Add score-related review reasons while allowing partial/cross-source evidence to soften warnings."""
    if score < 0.60:
        reasons.append(f"{prefix}_low_score")
    elif score < 0.72:
        if partial:
            reasons.append(f"{prefix}_partial_contingency_low_score")
        else:
            reasons.append(f"{prefix}_low_score")
    elif score < 0.88:
        # Do not downgrade a clearly confirmed row only because Dayzer uses a short facility alias.
        if not cross_confirmed:
            reasons.append(f"{prefix}_medium_score")


def add_contingency_reason(reasons: list[str], prefix: str, exact: bool, partial: bool) -> None:
    """Distinguish exact, partial, and mismatched contingency evidence."""
    if exact:
        return
    if partial:
        reasons.append(f"{prefix}_contingency_partial")
    else:
        reasons.append(f"{prefix}_contingency_mismatch")


def build_review_reason(row: pd.Series) -> str:
    """Explain why a row is not a clean high-confidence mapping."""
    reasons = []
    cross_confirmed = cross_source_confirmed_high(row)

    add_score_reason(
        reasons,
        "dayzer",
        row["dayzer_score"],
        bool(row["dayzer_contingency_partial"]) and not bool(row["dayzer_contingency_exact"]),
        cross_confirmed,
    )
    add_score_reason(
        reasons,
        "pano",
        row["pano_score"],
        bool(row["pano_contingency_partial"]) and not bool(row["pano_contingency_exact"]),
        cross_confirmed,
    )

    add_contingency_reason(
        reasons,
        "dayzer",
        bool(row["dayzer_contingency_exact"]),
        bool(row["dayzer_contingency_partial"]),
    )
    add_contingency_reason(
        reasons,
        "pano",
        bool(row["pano_contingency_exact"]),
        bool(row["pano_contingency_partial"]),
    )

    if row["dayzer_voltage_status"] == "mismatch":
        reasons.append("dayzer_voltage_mismatch")
    if row["pano_voltage_status"] == "mismatch":
        reasons.append("pano_voltage_mismatch")

    if bool(row["dayzer_duplicate_match"]):
        reasons.append("dayzer_duplicate_target")
    if bool(row["pano_duplicate_match"]):
        reasons.append("pano_duplicate_target")

    if bool(row["interface_flag"]):
        reasons.append("interface_style_constraint")

    if row["pano_validity_flag"] == "missing_validity_window":
        reasons.append("pano_validity_missing")

    if row.get("dayzer_match_status") == "unmatched_low_confidence":
        reasons.append("dayzer_unmatched_low_confidence")
    if row.get("pano_match_status") == "unmatched_low_confidence":
        reasons.append("pano_unmatched_low_confidence")
    if row.get("dayzer_match_status") == "needs_review":
        reasons.append("dayzer_candidate_needs_review")
    if row.get("pano_match_status") == "needs_review":
        reasons.append("pano_candidate_needs_review")

    if not reasons:
        if cross_confirmed:
            reasons.append("cross_source_confirmed_high")
        elif row["match_quality"] == "high":
            reasons.append("high_confidence_text_match")
        elif row["match_quality"] == "medium":
            reasons.append("initial_medium_quality")
        else:
            reasons.append("initial_review_quality")

    return join_reasons(reasons)


result["review_reason"] = result.apply(build_review_reason, axis=1)
result[["match_quality", "review_reason"]].head(10)


,match_quality,review_reason
0,high,cross_source_confirmed_high
1,medium,dayzer_partial_contingency_low_score;pano_medi...
2,high,dayzer_duplicate_target;pano_duplicate_target
3,high,cross_source_confirmed_high
4,medium,dayzer_medium_score;dayzer_contingency_partial
5,high,dayzer_medium_score;dayzer_duplicate_target
6,medium,dayzer_medium_score;dayzer_contingency_mismatc...
7,high,dayzer_duplicate_target;pano_duplicate_target
8,medium,dayzer_medium_score;dayzer_contingency_partial...
9,high,dayzer_duplicate_target


## Step 29: Assign Final Audited Match Quality

This cell creates `audited_match_quality`, which is the recommended final confidence field.

The final label is more conservative than `match_quality` because it uses the audit information. The intended meaning is:

| Final label | Meaning |
|---|---|
| `high` | strong text match and no major audit warning |
| `medium` | plausible match with one or more soft warnings |
| `review` | low score, mismatch, low-confidence candidate, or other issue requiring manual review |

The label does not delete uncertain matches. It keeps them visible while making their uncertainty explicit.

In [27]:
def assign_audited_quality(row: pd.Series) -> str:
    """Adjust match quality using audit flags and cross-source confirmation."""
    severe_reasons = {
        "dayzer_low_score",
        "pano_low_score",
        "dayzer_voltage_mismatch",
        "pano_voltage_mismatch",
        "pano_validity_missing",
        "dayzer_unmatched_low_confidence",
        "pano_unmatched_low_confidence",
    }
    downgrade_reasons = {
        "dayzer_contingency_mismatch",
        "pano_contingency_mismatch",
        "dayzer_duplicate_target",
        "pano_duplicate_target",
        "interface_style_constraint",
        "dayzer_medium_score",
        "pano_medium_score",
        "dayzer_partial_contingency_low_score",
        "pano_partial_contingency_low_score",
        "dayzer_contingency_partial",
        "pano_contingency_partial",
        "dayzer_candidate_needs_review",
        "pano_candidate_needs_review",
    }

    reasons = set(str(row["review_reason"]).split(";"))

    if reasons.intersection(severe_reasons):
        return "review"

    # If the only signal is strong cross-source confirmation, keep the row high.
    if cross_source_confirmed_high(row) and not reasons.intersection(downgrade_reasons):
        return "high"

    # Partial contingency evidence is plausible, but it should remain below clean high confidence.
    if row["match_quality"] == "review":
        partial_supported = (
            (bool(row["dayzer_contingency_partial"]) or bool(row["pano_contingency_partial"]))
            and max(row["dayzer_score"], row["pano_score"]) >= 0.75
        )
        return "medium" if partial_supported else "review"

    if row["match_quality"] == "medium" or reasons.intersection(downgrade_reasons):
        return "medium"

    return "high"


result["audited_match_quality"] = result.apply(assign_audited_quality, axis=1)
result[["match_quality", "audited_match_quality", "review_reason"]].head(10)


,match_quality,audited_match_quality,review_reason
0,high,high,cross_source_confirmed_high
1,medium,medium,dayzer_partial_contingency_low_score;pano_medi...
2,high,medium,dayzer_duplicate_target;pano_duplicate_target
3,high,high,cross_source_confirmed_high
4,medium,medium,dayzer_medium_score;dayzer_contingency_partial
5,high,medium,dayzer_medium_score;dayzer_duplicate_target
6,medium,medium,dayzer_medium_score;dayzer_contingency_mismatc...
7,high,medium,dayzer_duplicate_target;pano_duplicate_target
8,medium,medium,dayzer_medium_score;dayzer_contingency_partial...
9,high,medium,dayzer_duplicate_target


## Step 30: Meaning of `audited_match_quality`

Use `audited_match_quality` as the final confidence field in the submission.

The difference between the two quality fields is:

| Field | Meaning |
|---|---|
| `match_quality` | first-pass text similarity confidence |
| `audited_match_quality` | final confidence after contingency, voltage, duplicate, interface, validity, and source-side status checks |
| `review_reason` | explanation of why a row is not a clean high-confidence match |

This separation keeps the process transparent: the notebook shows both the raw matching confidence and the final conservative interpretation.

## Step 31: Select and Order Final Output Columns

This cell orders the output columns so the most important submission fields appear first.

The required fields are:

- `market_constraint`
- `dayzer_constraint`
- `pano_constraint`

Additional fields are included to make the result auditable: canonical ID, source-side statuses, similarity scores, contingency flags, voltage checks, duplicate flags, validity flags, and review reasons.

In [28]:
preferred_columns = [
    "canonical_constraint_id",
    "market_constraint",
    "dayzer_constraint",
    "dayzer_match_status",
    "dayzer_match_note",
    "pano_constraint",
    "pano_match_status",
    "pano_match_note",
    "match_quality",
    "audited_match_quality",
    "review_reason",
    "dayzer_match_quality",
    "pano_match_quality",
    "dayzer_score",
    "pano_score",
    "dayzer_contingency_exact",
    "dayzer_contingency_partial",
    "pano_contingency_exact",
    "pano_contingency_partial",
    "dayzer_voltage_status",
    "pano_voltage_status",
    "market_voltage_kv",
    "dayzer_voltage_kv",
    "pano_voltage_kv",
    "dayzer_duplicate_match",
    "pano_duplicate_match",
    "interface_flag",
    "market_interface_flag",
    "dayzer_interface_flag",
    "pano_interface_flag",
    "pano_validity_flag",
    "pano_earliest",
    "pano_latest",
    "market_constraint_id",
    "market_contingency_id",
    "dayzer_id",
    "pano_id",
    "market_facility_norm",
    "market_contingency_norm",
    "dayzer_facility_norm",
    "dayzer_contingency_norm",
    "pano_facility_norm",
    "pano_contingency_norm",
]

remaining_columns = [column for column in result.columns if column not in preferred_columns]
result = result[[column for column in preferred_columns if column in result.columns] + remaining_columns]

result.head(5)


,canonical_constraint_id,market_constraint,dayzer_constraint,dayzer_match_status,dayzer_match_note,pano_constraint,pano_match_status,pano_match_note,match_quality,audited_match_quality,...,market_constraint_id,market_contingency_id,dayzer_id,pano_id,market_facility_norm,market_contingency_norm,dayzer_facility_norm,dayzer_contingency_norm,pano_facility_norm,pano_contingency_norm
0,PJMISO_CANON_10002384305_10002493264,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV : L500.C...,NOTTINGH_230 KV_2-3:L500.Conastone-PeachBottom...,matched,NOTTINGH_230 KV_2-3:L500.Conastone-PeachBottom...,NOTTINGHM 2-3 SER DEV A 230 KV : L500.C...,matched,NOTTINGHM 2-3 SER DEV A 230 KV : L500.C...,high,high,...,10002384305,10002493264,532145,15915,NOTTINGHM 2 3 SER DEV A 230 KV NOTTINGH 230 KV...,L 500 CONASTONE PEACHBOTTOM 5012,NOTTINGH 230 KV 2 3,L 500 CONASTONE PEACHBOTTOM 5012,NOTTINGHM 2 3 SER DEV A 230 KV NOTTINGHM 2 3 S...,L 500 CONASTONE PEACHBOTTOM 5012
1,PJMISO_CANON_10000566138_10001865201,LENOX 115 KV LENOX-NMESHOPP NML 1090 : L230.ET...,LENOX_115 KV_LEN-NME:DBL:ETowanda-Hillside+Len...,partial_match,PARTIAL MATCH candidate: LENOX_115 KV_LEN-NME:...,NMESHOPP 115KV - LENOX 115KV (LENOX 115 KV LEN...,matched,NMESHOPP 115KV - LENOX 115KV (LENOX 115 KV LEN...,medium,medium,...,10000566138,10001865201,552352,783,LENOX NMESHOPP NML 1090 B 115 KV LENOX 115 KV ...,L 230 ETOWANDA HILLSIDE 2002 NYISO,LENOX 115 KV LEN NME,DBL ETOWANDA HILLSIDE LENOX NMESH,LENOX 115 KV LEN NME NMESHOPP 115 KV LENOX 115...,L 230 ETOWANDA HILLSIDE 2002 NYISO
2,PJMISO_CANON_10004072634_10000680485,EASTON 69 KV EAS-EMU : ACTUAL,EASTON_69 KV_EAS-EMU:ACTUAL,matched,EASTON_69 KV_EAS-EMU:ACTUAL,EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU...,matched,EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU...,high,medium,...,10004072634,10000680485,520100,1920,EASTON 69 KV EAS EMU EASTON 69 KV EAS EMU,ACTUAL,EASTON 69 KV EAS EMU,ACTUAL,EASTON 69 KV EAS EMU EASTON 69 KV EMUNI 69 KV ...,ACTUAL
3,PJMISO_CANON_10001822928_10000680485,SAYRECON230 KV SAY-SAY : ACTUAL,SAYRECON_230 KV_SAY-SAY:ACTUAL,matched,SAYRECON_230 KV_SAY-SAY:ACTUAL,SAYREVIL 230KV - SAYRECON 230KV (SAYRECON 230 ...,matched,SAYREVIL 230KV - SAYRECON 230KV (SAYRECON 230 ...,high,high,...,10001822928,10000680485,512652,2035,SAYRECON 230 KV SAY SAY SAYRECON 230 KV SAY SAY,ACTUAL,SAYRECON 230 KV SAY SAY,ACTUAL,SAYRECON 230 KV SAY SAY SAYREVIL 230 KV SAYREC...,ACTUAL
4,PJMISO_CANON_10000465752_10016012236,MOUN UGI 230 KV MOUN UGI 2 XFORMER : 230/66.MO...,MOUN UGI_230 KV_1:230/66.Mountain.T2 (Sctnlz),partial_match,PARTIAL MATCH candidate: MOUN UGI_230 KV_1:230...,MOUN UGI 69KV - MOUN UGI 230KV (MOUN UGI 230 K...,matched,MOUN UGI 69KV - MOUN UGI 230KV (MOUN UGI 230 K...,medium,medium,...,10000465752,10016012236,550634,1060,MOUN UGI 230 KV 2 MOUN UGI 230 KV MOUN UGI 2 X...,230 66 MOUNTAIN T 1 SCTNLZ,MOUN UGI 230 KV 1,230 66 MOUNTAIN T 2 SCTNLZ,MOUN UGI 230 KV 2 MOUN UGI 69 KV MOUN UGI 230 ...,230 66 MOUNTAIN T 1 SCTNLZ


## Step 32: Summarize Match Results

This cell prints the main result statistics.

The summary checks whether the output has the expected number of rows and shows how many rows fall into each quality group. It also reports the number of exact and partial contingency matches, source-side status counts, duplicate target counts, voltage warnings, and common review reasons.

In [29]:
quality_counts = result["match_quality"].value_counts().reindex(["high", "medium", "review"], fill_value=0)
audited_counts = result["audited_match_quality"].value_counts().reindex(["high", "medium", "review"], fill_value=0)
dayzer_quality_counts = result["dayzer_match_quality"].value_counts().reindex(["high", "medium", "review"], fill_value=0)
pano_quality_counts = result["pano_match_quality"].value_counts().reindex(["high", "medium", "review"], fill_value=0)
reason_counts = result["review_reason"].str.get_dummies(sep=";").sum().sort_values(ascending=False)
dayzer_status_counts = result["dayzer_match_status"].value_counts().reindex(["matched", "partial_match", "needs_review", "unmatched_low_confidence"], fill_value=0)
pano_status_counts = result["pano_match_status"].value_counts().reindex(["matched", "partial_match", "needs_review", "unmatched_low_confidence"], fill_value=0)

dayzer_exact_count = int(result["dayzer_contingency_exact"].sum())
pano_exact_count = int(result["pano_contingency_exact"].sum())
dayzer_partial_count = int((result["dayzer_contingency_partial"] & ~result["dayzer_contingency_exact"]).sum())
pano_partial_count = int((result["pano_contingency_partial"] & ~result["pano_contingency_exact"]).sum())
cross_confirmed_count = int(result.apply(cross_source_confirmed_high, axis=1).sum())
total_rows = len(result)

print(f"Output rows: {total_rows:,}")
print("\nInitial match_quality:")
display(quality_counts.to_frame("rows"))

print("\nFinal audited_match_quality:")
display(audited_counts.to_frame("rows"))

print("\nDayzer side quality:")
display(dayzer_quality_counts.to_frame("rows"))

print("\nPanorama side quality:")
display(pano_quality_counts.to_frame("rows"))

print("\nSource-side match status:")
display(pd.DataFrame({"dayzer_rows": dayzer_status_counts, "pano_rows": pano_status_counts}))

print("\nContingency agreement:")
display(pd.DataFrame({
    "rows": {
        "dayzer_exact": dayzer_exact_count,
        "dayzer_partial_only": dayzer_partial_count,
        "pano_exact": pano_exact_count,
        "pano_partial_only": pano_partial_count,
        "cross_source_confirmed_high_candidates": cross_confirmed_count,
    }
}))

print("\nMost common review reasons:")
display(reason_counts.head(15).to_frame("rows"))


Output rows: 5,230

Initial match_quality:


,rows
match_quality,
high,4415
medium,634
review,181



Final audited_match_quality:


,rows
audited_match_quality,
high,2728
medium,2204
review,298



Dayzer side quality:


...

,dayzer_rows,pano_rows
matched,4424,4972
partial_match,295,90
needs_review,441,149
unmatched_low_confidence,70,19



Contingency agreement:


,rows
dayzer_exact,4500
dayzer_partial_only,297
pano_exact,5013
pano_partial_only,92
cross_source_confirmed_high_candidates,3159



Most common review reasons:


,rows
cross_source_confirmed_high,2267
dayzer_duplicate_target,1697
pano_duplicate_target,1060
dayzer_medium_score,832
high_confidence_text_match,461
dayzer_candidate_needs_review,441
dayzer_contingency_mismatch,433
pano_medium_score,336
dayzer_contingency_partial,297
dayzer_low_score,211


## Step 33: Inspect Example Matches

This cell displays examples from high, medium, and review groups.

Manual inspection is useful because text matching can behave differently across ordinary line constraints, interface constraints, and complex Dayzer contingency strings. The examples help verify whether the quality labels are reasonable.

In [30]:
example_columns = [
    "canonical_constraint_id",
    "market_constraint",
    "dayzer_constraint",
    "dayzer_match_status",
    "pano_constraint",
    "pano_match_status",
    "match_quality",
    "audited_match_quality",
    "review_reason",
    "dayzer_score",
    "pano_score",
    "dayzer_contingency_exact",
    "dayzer_contingency_partial",
    "pano_contingency_exact",
    "pano_contingency_partial",
]

print("High examples")
display(result[result["audited_match_quality"] == "high"][example_columns].head(5))

print("Medium examples")
display(result[result["audited_match_quality"] == "medium"][example_columns].head(5))

print("Review examples")
display(result[result["audited_match_quality"] == "review"][example_columns].head(5))


High examples


,canonical_constraint_id,market_constraint,dayzer_constraint,dayzer_match_status,pano_constraint,pano_match_status,match_quality,audited_match_quality,review_reason,dayzer_score,pano_score,dayzer_contingency_exact,dayzer_contingency_partial,pano_contingency_exact,pano_contingency_partial
0,PJMISO_CANON_10002384305_10002493264,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV : L500.C...,NOTTINGH_230 KV_2-3:L500.Conastone-PeachBottom...,matched,NOTTINGHM 2-3 SER DEV A 230 KV : L500.C...,matched,high,high,cross_source_confirmed_high,0.7240,0.9755,True,False,True,False
3,PJMISO_CANON_10001822928_10000680485,SAYRECON230 KV SAY-SAY : ACTUAL,SAYRECON_230 KV_SAY-SAY:ACTUAL,matched,SAYREVIL 230KV - SAYRECON 230KV (SAYRECON 230 ...,matched,high,high,cross_source_confirmed_high,0.9902,0.9678,True,False,True,False
10,PJMISO_CANON_10015848492_10016636204,RAMAPO 138 KV RAM-SMA : 345/138.SOUTHMAHWAH....,RAMAPO_138 KV_RAM-SMA:345/138.SouthMahwah.BK258,matched,RAMAPO 138KV - SMAWA 138KV (RAMAPO 138 KV RAM-...,matched,high,high,cross_source_confirmed_high,0.9478,0.9670,True,False,True,False
12,PJMISO_CANON_10002848522_10002903265,94 HAURD-11323 11323 B 138 KV : L345.NELSON-EL...,94 HAURD_138 KV_11323:L345.Nelson-Electric Jun...,matched,94 HAURD-11323 11323 B 138 KV : L345.N...,matched,high,high,cross_source_confirmed_high,0.9011,0.9895,True,False,True,False
24,PJMISO_CANON_10000466153_10002493264,TMI 500 KV TMI 1 BANK XFORMER : L500.CONASTONE...,TMI_500 KV_1 BANK:L500.Conastone-PeachBottom.5012,matched,TMI 500KV - TMI 230KV (TMI 500 KV 1 BANK) : L5...,matched,high,high,cross_source_confirmed_high,0.8773,0.9633,True,False,True,False


Medium examples


,canonical_constraint_id,market_constraint,dayzer_constraint,dayzer_match_status,pano_constraint,pano_match_status,match_quality,audited_match_quality,review_reason,dayzer_score,pano_score,dayzer_contingency_exact,dayzer_contingency_partial,pano_contingency_exact,pano_contingency_partial
1,PJMISO_CANON_10000566138_10001865201,LENOX 115 KV LENOX-NMESHOPP NML 1090 : L230.ET...,LENOX_115 KV_LEN-NME:DBL:ETowanda-Hillside+Len...,partial_match,NMESHOPP 115KV - LENOX 115KV (LENOX 115 KV LEN...,matched,medium,medium,dayzer_partial_contingency_low_score;pano_medi...,0.7130,0.8041,False,True,True,False
2,PJMISO_CANON_10004072634_10000680485,EASTON 69 KV EAS-EMU : ACTUAL,EASTON_69 KV_EAS-EMU:ACTUAL,matched,EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU...,matched,high,medium,dayzer_duplicate_target;pano_duplicate_target,0.9781,0.9469,True,False,True,False
4,PJMISO_CANON_10000465752_10016012236,MOUN UGI 230 KV MOUN UGI 2 XFORMER : 230/66.MO...,MOUN UGI_230 KV_1:230/66.Mountain.T2 (Sctnlz),partial_match,MOUN UGI 69KV - MOUN UGI 230KV (MOUN UGI 230 K...,matched,medium,medium,dayzer_medium_score;dayzer_contingency_partial,0.8758,0.9716,False,True,True,False
5,PJMISO_CANON_10000565711_10002493264,GRACETON 230 KV GRACETON-SAFEHARB 2303 : L500....,GRACETON_230 KV_GRA-MANO:L500.Conastone-PeachB...,matched,GRACETON230 KV GRA-SAF : L500.Conastone-Peach...,matched,high,medium,dayzer_medium_score;dayzer_duplicate_target,0.7812,0.9114,True,False,True,False
6,PJMISO_CANON_10002848522_10017401059,94 HAURD-11323 11323 B 138 KV : L345.NELSON-EL...,94 HAURD_138 KV_11323:ACTUAL,needs_review,94 HAURD 138KV - 11323 138KV (94 HAURD 138 KV ...,matched,medium,medium,dayzer_medium_score;dayzer_contingency_mismatc...,0.7420,0.9560,False,False,True,False


Review examples


,canonical_constraint_id,market_constraint,dayzer_constraint,dayzer_match_status,pano_constraint,pano_match_status,match_quality,audited_match_quality,review_reason,dayzer_score,pano_score,dayzer_contingency_exact,dayzer_contingency_partial,pano_contingency_exact,pano_contingency_partial
13,PJMISO_CANON_10000565443_10000680485,BERWICK 69 KV BERWICK-KOONSVIL : ACTUAL,MOUN UGI_230 KV_3:L69.Hunlock-Koonsville-Berwick,needs_review,BERWICK 69 KV BER-KOO : BASE,matched,review,review,dayzer_low_score;dayzer_contingency_mismatch;d...,0.6487,0.9050,False,False,True,False
32,PJMISO_CANON_10017408419_10017408672,PREST - TIBBS 138 KV L/O ASTER - COMMODORE 345...,PRES_138 KV_PRE-TIBB:Aster - CMDR 345 kV,needs_review,PRES 138KV - TIBB 138KV (PRES 138 KV PRE-TIBB)...,unmatched_low_confidence,review,review,dayzer_low_score;pano_low_score;pano_contingen...,0.6544,0.5821,True,False,False,False
40,PJMISO_CANON_10000567660_10004059893,177 BURN 345 KV BURNHAM-MUNSTER2 TIE : L765.DU...,MUNSTER2_345 KV_A:L765.Dumont-Wilton Center.11215,matched,Burnham Munster l/o Dumont Wilton Center : DUM...,partial_match,medium,review,dayzer_medium_score;pano_contingency_partial;p...,0.8457,0.9553,True,False,False,True
48,PJMISO_CANON_10016050327_10015883528,PA-CENT CONTINGENCY 1 : L500.JUNIATA-SUNBURY.5046,JUNIATA_500 KV_2:L500.Juniata-Sunbury.5046,needs_review,JUNIATA 500 KV 2 : JUNIATA-SUNBURY (5046) 500...,unmatched_low_confidence,review,review,dayzer_low_score;pano_low_score;pano_contingen...,0.6272,0.5925,True,False,False,True
50,PJMISO_CANON_10016736777_10016033675,SUB85_SUB18_161_FLO_OAKGROVE_LOUISA_345 : L345...,SB 18 6_161 KV_LN 4447:OakGrove - Louisa 345 kV,partial_match,SB 85 56 161KV - ROCK IS 161KV (ROCK IS 161 KV...,unmatched_low_confidence,review,review,dayzer_medium_score;pano_low_score;dayzer_cont...,0.7586,0.5424,False,True,True,False


## Step 34: Export Matched CSV

This cell saves the final matched table to CSV.

The CSV contains the required matched constraint columns plus additional confidence and audit fields. The extra fields are included so a reviewer can understand which matches are clean and which ones need manual validation.

In [ ]:
result.to_csv(MATCHED_CSV, index=False)
print(f"Saved matched CSV: {MATCHED_CSV}")


## Step 35: Generate Summary Report

 It summarizes the main design choices and explains how to interpret `match_quality`, `audited_match_quality`, `source-side status`, and `review_reason`.

## Step 36: Final Submission Checklist

Before submission:

1. Restart the kernel and run all cells from top to bottom.
2. Confirm that the matched CSV is generated.
3. Confirm that the summary report is generated.
4. Submit the notebook, matched CSV, and report together.
5. In the report and discussion, use `audited_match_quality` as the final confidence field.
6. Treat `review` rows as manual-validation candidates rather than automatic failures.